# Credit Analysis ML Web App

## Overview

This project uses machine learning to identify serious loan delinquency with Logistic Regression. It includes a FastAPI backend and an HTML frontend, allowing users to interact with the model through a web page.

## Requirements

Install the required libraries by running this command:

```bash
pip install pandas numpy scikit-learn joblib fastapi uvicorn
```

## How to Run

1. Open `Credit_Analysis.ipynb` in Google Colab or Jupyter Notebook.
2. Upload the `cs-training.csv` dataset to the location specified in the notebook.
3. Run the notebook cells in order to prepare the data, train the model, and save it.
4. Run the relevant notebook cells to create `main.py` and `index.html`.
5. Start the FastAPI backend by running:

   ```bash
   uvicorn main:app --reload
   ```

6. Open `http://127.0.0.1:8000` in your browser to access the web app.

## Features

- Classifies serious loan delinquency
- Uses FastAPI for the backend
- Provides an HTML user interface

## Note

Keep the trained model files, `main.py`, and `index.html` in the same project folder. Make sure the dataset path matches your setup so the notebook can find the file.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df=pd.read_csv(r"/content/cs-training.csv")
df.isnull().sum()

,0
Unnamed: 0,0
SeriousDlqin2yrs,0
RevolvingUtilizationOfUnsecuredLines,0
age,0
NumberOfTime30-59DaysPastDueNotWorse,0
DebtRatio,0
MonthlyIncome,29731
NumberOfOpenCreditLinesAndLoans,0
NumberOfTimes90DaysLate,0
NumberRealEstateLoansOrLines,0


In [ ]:
df.shape

(150000, 12)

In [ ]:
df.columns

Index(['Unnamed: 0', 'SeriousDlqin2yrs',
       'RevolvingUtilizationOfUnsecuredLines', 'age',
       'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome',
       'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate',
       'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse',
       'NumberOfDependents'],
      dtype='object')

In [ ]:
df.describe()

,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
count,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,1.202690e+05,150000.000000,150000.000000,150000.000000,150000.000000,146076.000000
mean,75000.500000,0.066840,6.048438,52.295207,0.421033,353.005076,6.670221e+03,8.452760,0.265973,1.018240,0.240387,0.757222
std,43301.414527,0.249746,249.755371,14.771866,4.192781,2037.818523,1.438467e+04,5.145951,4.169304,1.129771,4.155179,1.115086
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
25%,37500.750000,0.000000,0.029867,41.000000,0.000000,0.175074,3.400000e+03,5.000000,0.000000,0.000000,0.000000,0.000000
50%,75000.500000,0.000000,0.154181,52.000000,0.000000,0.366508,5.400000e+03,8.000000,0.000000,1.000000,0.000000,0.000000
75%,112500.250000,0.000000,0.559046,63.000000,0.000000,0.868254,8.249000e+03,11.000000,0.000000,2.000000,0.000000,1.000000
max,150000.000000,1.000000,50708.000000,109.000000,98.000000,329664.000000,3.008750e+06,58.000000,98.000000,54.000000,98.000000,20.000000


In [ ]:
df["MonthlyIncome"]=df["MonthlyIncome"].fillna(df["MonthlyIncome"].median())

In [ ]:
df["NumberOfDependents"]=df["NumberOfDependents"].fillna(df["NumberOfDependents"].mode()[0])

In [ ]:
df.isnull().sum()

,0
Unnamed: 0,0
SeriousDlqin2yrs,0
RevolvingUtilizationOfUnsecuredLines,0
age,0
NumberOfTime30-59DaysPastDueNotWorse,0
DebtRatio,0
MonthlyIncome,0
NumberOfOpenCreditLinesAndLoans,0
NumberOfTimes90DaysLate,0
NumberRealEstateLoansOrLines,0


Outlier handling

In [ ]:
upper_limit = df['MonthlyIncome'].quantile(.99)
lower_limit = df['MonthlyIncome'].quantile(.01)
df = df[(df['MonthlyIncome'] < upper_limit) &(df['MonthlyIncome'] > lower_limit)]
df['MonthlyIncome']

,MonthlyIncome
0,9120.0
1,2600.0
2,3042.0
3,3300.0
5,3500.0
...,...
149995,2100.0
149996,5584.0
149997,5400.0
149998,5716.0


In [ ]:
import pandas as pd
import numpy as np

df.replace('NA', np.nan, inplace=True)
df= df.apply(pd.to_numeric, errors='coerce')

# Store all numeric columns at once
numeric_df = df.select_dtypes(include=[np.number])

# Calculate Q1, Q3, IQR for ALL columns at once
Q1 = numeric_df.quantile(0.25)
Q3 = numeric_df.quantile(0.75)
IQR = Q3 - Q1

upper_limit = Q3 + 1.5 * IQR
lower_limit = Q1 - 1.5 * IQR

print("Upper Limits:\n", upper_limit)
print("\nLower Limits:\n", lower_limit)

# Count outliers for ALL columns at once
outliers_count = ((numeric_df > upper_limit) | (numeric_df < lower_limit)).sum()
print("\nTotal Extreme Transactions:\n", outliers_count)

# Clip ALL columns at once
clipped_df = numeric_df.clip(lower_limit, upper_limit, axis=1)

print("\nMaximum after clipping:\n", clipped_df.max())
print("\nMinimum after clipping:\n", clipped_df.min())

Upper Limits:
 Unnamed: 0                              224960.000000
SeriousDlqin2yrs                             0.000000
RevolvingUtilizationOfUnsecuredLines         1.356528
age                                         96.000000
NumberOfTime30-59DaysPastDueNotWorse         0.000000
DebtRatio                                    1.817961
MonthlyIncome                            12222.500000
NumberOfOpenCreditLinesAndLoans             20.000000
NumberOfTimes90DaysLate                      0.000000
NumberRealEstateLoansOrLines                 5.000000
NumberOfTime60-89DaysPastDueNotWorse         0.000000
NumberOfDependents                           2.500000
dtype: float64

Lower Limits:
 Unnamed: 0                             -74952.000000
SeriousDlqin2yrs                            0.000000
RevolvingUtilizationOfUnsecuredLines       -0.766102
age                                         8.000000
NumberOfTime30-59DaysPastDueNotWorse        0.000000
DebtRatio                                

# Deployment: ML Models + FastAPI Backend + HTML Frontend
The cells below train a **linear regression** (predicts `MonthlyIncome`, reports R²) and a **classification model** (logistic regression, predicts `SeriousDlqin2yrs`, reports accuracy), save both with `joblib`, then write the FastAPI backend (`main.py`) and the HTML page (`index.html`).

## 1. Train and save the models

In [ ]:

import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score, accuracy_score

# 1. Load dataset


# 2. Remove unnecessary columns
df = df.drop(columns=["Id"], errors="ignore")

# 3. Handle missing values
df = df.replace("NA", float("nan"))

df = df.apply(pd.to_numeric, errors="coerce")

df["NumberOfDependents"] = df["NumberOfDependents"].fillna(
    df["NumberOfDependents"].mode()[0]
)

df = df.fillna(df.median())

# 4. Select input features
features = [
    "RevolvingUtilizationOfUnsecuredLines",
    "age",
    "NumberOfTime30-59DaysPastDueNotWorse",
    "DebtRatio",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberOfTimes90DaysLate",
    "NumberRealEstateLoansOrLines",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfDependents"
]

# 5. LINEAR REGRESSION
X = df[features]
y = df["MonthlyIncome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

reg_model = LinearRegression()
reg_model.fit(X_train, y_train)

predictions = reg_model.predict(X_test)

print("Regression R2 Score:", r2_score(y_test, predictions))

# 6. LOGISTIC REGRESSION
X = df[features + ["MonthlyIncome"]]
y = df["SeriousDlqin2yrs"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

clf_model.fit(X_train, y_train)

predictions = clf_model.predict(X_test)

print("Classification Accuracy:", accuracy_score(y_test, predictions))

# 7. Save models
joblib.dump(reg_model, "regression_model.joblib")
joblib.dump(clf_model, "classification_model.joblib")

print("Models saved successfully!")


Regression R2 Score: 0.04194315147418759
Classification Accuracy: 0.934
Models saved successfully!


## 2. Backend: FastAPI (`main.py`)
Two POST endpoints: `/predict/regression` and `/predict/classification`.

In [ ]:
%%writefile main.py
from pathlib import Path
import joblib
import pandas as pd
from fastapi import FastAPI
from fastapi.responses import FileResponse

# 1. Load trained models (saved directly in the current directory)
BASE = Path(".").resolve()

reg_model = joblib.load(BASE / "regression_model.joblib")
clf_model = joblib.load(BASE / "classification_model.joblib")

app = FastAPI()

# 2. Input column names
features = [
    "RevolvingUtilizationOfUnsecuredLines",
    "age",
    "NumberOfTime30-59DaysPastDueNotWorse",
    "DebtRatio",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberOfTimes90DaysLate",
    "NumberRealEstateLoansOrLines",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfDependents"
]

# 3. Regression prediction
@app.post("/predict/regression")
def predict_regression(data: dict):
    values = [
        data["revolving_utilization"],
        data["age"],
        data["past_due_30_59"],
        data["debt_ratio"],
        data["open_credit_lines"],
        data["times_90_days_late"],
        data["real_estate_loans"],
        data["past_due_60_89"],
        data["dependents"]
    ]
    X = pd.DataFrame([values], columns=features)
    prediction = reg_model.predict(X)[0]

    return {
        "prediction": round(max(float(prediction), 0), 2),
        "model_r2": "0.04"
    }

# 4. Classification prediction
@app.post("/predict/classification")
def predict_classification(data: dict):
    clf_features = features + ["MonthlyIncome"]
    values = [
        data["revolving_utilization"],
        data["age"],
        data["past_due_30_59"],
        data["debt_ratio"],
        data["open_credit_lines"],
        data["times_90_days_late"],
        data["real_estate_loans"],
        data["past_due_60_89"],
        data["dependents"],
        data["monthly_income"]
    ]
    X = pd.DataFrame([values], columns=clf_features)
    prediction = clf_model.predict(X)[0]
    probabilities = clf_model.predict_proba(X)[0] if hasattr(clf_model, 'predict_proba') else [0.0, 0.0]

    return {
        "prediction": int(prediction),
        "label": "Likely to default" if prediction == 1 else "Unlikely to default",
        "default_probability": float(probabilities[1]),
        "model_accuracy": 0.934
    }

# 5. Open HTML page
@app.get("/")
def home():
    return FileResponse(BASE / "index.html")


Overwriting main.py


## 3. Frontend: HTML page (`index.html`)
One form per model; each sends its input to the backend and shows the result.

In [ ]:
%%writefile index.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Credit ML Predictor</title>
<style>
  :root { --bg:#f4f6fb; --card:#fff; --text:#1f2937; --muted:#6b7280; --accent:#2563eb;
          --border:#d1d5db; --ok:#15803d; --bad:#b91c1c; }
  @media (prefers-color-scheme: dark) {
    :root { --bg:#0f172a; --card:#1e293b; --text:#e5e7eb; --muted:#94a3b8; --accent:#3b82f6;
            --border:#475569; --ok:#4ade80; --bad:#f87171; }
  }
  * { box-sizing: border-box; }
  body { margin:0; font-family: system-ui, Arial, sans-serif; background:var(--bg); color:var(--text); }
  header { padding:28px 20px 8px; text-align:center; }
  header p { color:var(--muted); margin:6px 0 0; }
  main { max-width:1000px; margin:0 auto; padding:20px; display:grid; gap:24px;
         grid-template-columns:repeat(auto-fit, minmax(380px, 1fr)); }
  section { background:var(--card); border-radius:12px; padding:22px; box-shadow:0 1px 4px rgba(0,0,0,.12); }
  h2 { margin-top:0; font-size:1.2rem; }
  .grid { display:grid; grid-template-columns:1fr 1fr; gap:12px; }
  label { display:flex; flex-direction:column; font-size:.8rem; color:var(--muted); gap:4px; }
  input { padding:8px; border:1px solid var(--border); border-radius:6px; font-size:.95rem;
          background:transparent; color:var(--text); }
  button { margin-top:16px; width:100%; padding:11px; border:0; border-radius:8px;
           background:var(--accent); color:#fff; font-size:1rem; cursor:pointer; }
  button:disabled { opacity:.6; cursor:wait; }
  .result { margin-top:16px; padding:14px; border-radius:8px; border:1px dashed var(--border);
            min-height:52px; font-size:1.05rem; }
  .result small { display:block; color:var(--muted); margin-top:4px; font-size:.8rem; }
  .ok { color:var(--ok); } .bad { color:var(--bad); }
  @media (max-width:480px){ main{grid-template-columns:1fr;} .grid{grid-template-columns:1fr;} }
</style>
</head>
<body>
<header>
  <h1>Credit ML Predictor</h1>
  <p>Linear regression (income) &amp; logistic regression (default risk)</p>
</header>

<main>
  <!-- ============ REGRESSION FORM ============ -->
  <section>
    <h2>1. Predict Monthly Income (Regression)</h2>
    <form id="regForm">
      <div class="grid">
        <label>Revolving utilization<input name="revolving_utilization" type="number" step="any" min="0" value="0.3" required></label>
        <label>Age<input name="age" type="number" min="18" max="120" value="45" required></label>
        <label>Times 30-59 days past due<input name="past_due_30_59" type="number" min="0" value="0" required></label>
        <label>Debt ratio<input name="debt_ratio" type="number" step="any" min="0" value="0.4" required></label>
        <label>Open credit lines &amp; loans<input name="open_credit_lines" type="number" min="0" value="8" required></label>
        <label>Times 90+ days late<input name="times_90_days_late" type="number" min="0" value="0" required></label>
        <label>Real estate loans<input name="real_estate_loans" type="number" min="0" value="1" required></label>
        <label>Times 60-89 days past due<input name="past_due_60_89" type="number" min="0" value="0" required></label>
        <label>Dependents<input name="dependents" type="number" min="0" value="1" required></label>
      </div>
      <button type="submit">Predict income</button>
    </form>
    <div class="result" id="regResult">Result will appear here.</div>
  </section>

  <!-- ============ CLASSIFICATION FORM ============ -->
  <section>
    <h2>2. Predict Default Risk (Classification)</h2>
    <form id="clfForm">
      <div class="grid">
        <label>Revolving utilization<input name="revolving_utilization" type="number" step="any" min="0" value="0.3" required></label>
        <label>Age<input name="age" type="number" min="18" max="120" value="45" required></label>
        <label>Times 30-59 days past due<input name="past_due_30_59" type="number" min="0" value="0" required></label>
        <label>Debt ratio<input name="debt_ratio" type="number" step="any" min="0" value="0.4" required></label>
        <label>Monthly income<input name="monthly_income" type="number" step="any" min="0" value="5400" required></label>
        <label>Open credit lines &amp; loans<input name="open_credit_lines" type="number" min="0" value="8" required></label>
        <label>Times 90+ days late<input name="times_90_days_late" type="number" min="0" value="0" required></label>
        <label>Real estate loans<input name="real_estate_loans" type="number" min="0" value="1" required></label>
        <label>Times 60-89 days past due<input name="past_due_60_89" type="number" min="0" value="0" required></label>
        <label>Dependents<input name="dependents" type="number" min="0" value="1" required></label>
      </div>
      <button type="submit">Predict risk</button>
    </form>
    <div class="result" id="clfResult">Result will appear here.</div>
  </section>
</main>

<script>
  // Same origin when served by FastAPI; fall back to localhost if opened as a file.
  const API = location.protocol === "file:" ? "http://127.0.0.1:8000" : "";

  function wire(formId, resultId, endpoint, render) {
    const form = document.getElementById(formId);
    const box  = document.getElementById(resultId);
    form.addEventListener("submit", async (e) => {
      e.preventDefault();
      const btn = form.querySelector("button");
      const body = {};
      new FormData(form).forEach((v, k) => body[k] = Number(v));
      btn.disabled = true; box.textContent = "Predicting…";
      try {
        const res = await fetch(API + endpoint, {
          method: "POST",
          headers: { "Content-Type": "application/json" },
          body: JSON.stringify(body),
        });
        if (!res.ok) throw new Error("Server error " + res.status);
        box.innerHTML = render(await res.json());
      } catch (err) {
        box.innerHTML = `<span class="bad">Could not get a prediction: ${err.message}.
          Is the backend running?</span>`;
      } finally { btn.disabled = false; }
    });
  }

  wire("regForm", "regResult", "/predict/regression", d =>
    `Predicted monthly income: <b>$${d.prediction.toLocaleString()}</b>
     <small>Model R² on test data: ${d.model_r2}</small>`);

  wire("clfForm", "clfResult", "/predict/classification", d =>
    `<b class="${d.prediction ? "bad" : "ok"}">${d.label}</b>
     (class ${d.prediction})
     <small>Default probability: ${(d.default_probability * 100).toFixed(1)}% ·
            Model accuracy on test data: ${(d.model_accuracy * 100).toFixed(1)}%</small>`);
</script>
</body>
</html>


Overwriting index.html


In [ ]:
!pip install -q fastapi uvicorn joblib scikit-learn pandas nest_asyncio

import threading, time, uvicorn, nest_asyncio
nest_asyncio.apply()

import importlib, main
importlib.reload(main)

config = uvicorn.Config(main.app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)

# Run the server in a background thread so the cell doesn't block
threading.Thread(target=server.run, daemon=True).start()
time.sleep(3)  # give it a moment to start

# Expose the port through Colab's proxy
from google.colab import output
output.serve_kernel_port_as_iframe(8000, height=900)

INFO:     Started server process [655]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


<IPython.core.display.Javascript object>

In [ ]:
from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8000)")
print(url)

https://8000-m-s-use1b11-3t7qdluscaqt7-b.us-east1.prod.r.colab.dev
